# NLTK Capabilities Case Study
## Classical NLP on Customer-Support Text

This notebook illustrates the major capabilities of **NLTK (Natural Language Toolkit)** using one coherent case study: analysing customer-support messages for a fictional financial-technology company.

### Topics covered
- Tokenization and sentence segmentation
- Stopword removal
- Stemming and lemmatization
- POS tagging
- Chunking / shallow parsing
- Named Entity Recognition
- Context-Free Grammars
- Constituency parsing
- Dependency-related structures
- Probabilistic CFGs and Viterbi parsing
- WordNet / lexical semantics
- Word Sense Disambiguation
- N-grams
- Frequency and conditional-frequency analysis
- Collocations
- Statistical language modelling and smoothing
- Corpus processing
- Concordance and lexical exploration
- VADER sentiment analysis
- Edit distance
- Classical text classification
- Classical sequence tagging
- Evaluation metrics
- BLEU and METEOR

> NLTK is especially strong for **classical NLP, computational linguistics, corpus analysis, grammar/parsing, lexical semantics, and learning NLP internals**.

## 1. Setup

In [2]:
import nltk

resources = [
    "punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
    "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker", "maxent_ne_chunker_tab", "words",
    "vader_lexicon", "brown", "reuters", "gutenberg",
    "treebank", "movie_reviews"
]

for resource in resources:
    try:
        nltk.download(resource, quiet=True)
    except Exception as exc:
        print(f"Could not download {resource}: {exc}")

print("NLTK version:", nltk.__version__)

NLTK version: 3.9.2


In [5]:
from collections import Counter
from pprint import pprint

from nltk import (
    FreqDist, ConditionalFreqDist, Text,
    bigrams, trigrams, ngrams,
    pos_tag, ne_chunk, word_tokenize, sent_tokenize
)
from nltk.corpus import stopwords, wordnet as wn
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer, MWETokenizer
from nltk.chunk import RegexpParser
from nltk.parse import ChartParser, ViterbiParser, DependencyGraph
from nltk.grammar import CFG, PCFG
from nltk.wsd import lesk
from nltk.collocations import BigramCollocationFinder, TrigramCollocationFinder
from nltk.metrics import BigramAssocMeasures, TrigramAssocMeasures
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.lm import MLE, Laplace, KneserNeyInterpolated
from nltk.lm.preprocessing import padded_everygram_pipeline

## 2. Case-study corpus

In [6]:
documents = [
    {"id": 1, "label": "payment",
     "text": "Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice."},
    {"id": 2, "label": "account",
     "text": "Priya cannot access her savings account after resetting the password."},
    {"id": 3, "label": "fraud",
     "text": "Daniel noticed an unauthorized transaction from London and wants the card blocked immediately."},
    {"id": 4, "label": "payment",
     "text": "The mobile payment worked after I restarted the app. Excellent support!"},
    {"id": 5, "label": "account",
     "text": "My bank account is locked and login verification keeps failing."},
    {"id": 6, "label": "fraud",
     "text": "Someone used my credit card online. I did not make this purchase."},
    {"id": 7, "label": "payment",
     "text": "The transfer to Singapore was completed successfully, but the merchant has not received it."},
    {"id": 8, "label": "account",
     "text": "Password reset was quick and the account is accessible again."},
    {"id": 9, "label": "fraud",
     "text": "A suspicious card transaction appeared on my statement this morning."},
]

raw_text = " ".join(doc["text"] for doc in documents)
print(raw_text)

Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice. Priya cannot access her savings account after resetting the password. Daniel noticed an unauthorized transaction from London and wants the card blocked immediately. The mobile payment worked after I restarted the app. Excellent support! My bank account is locked and login verification keeps failing. Someone used my credit card online. I did not make this purchase. The transfer to Singapore was completed successfully, but the merchant has not received it. Password reset was quick and the account is accessible again. A suspicious card transaction appeared on my statement this morning.


# Part I — Preprocessing
## 3. Sentence and word tokenization

In [10]:
sentences = sent_tokenize(raw_text)
print("sentence token:",sentences)
print("Sentence count:", len(sentences))

tokens = word_tokenize(documents[0]["text"])
print("word tokens",tokens)

sentence token: ['Arjun paid SGD 120 at Marina Bay yesterday, but the card payment was declined twice.', 'Priya cannot access her savings account after resetting the password.', 'Daniel noticed an unauthorized transaction from London and wants the card blocked immediately.', 'The mobile payment worked after I restarted the app.', 'Excellent support!', 'My bank account is locked and login verification keeps failing.', 'Someone used my credit card online.', 'I did not make this purchase.', 'The transfer to Singapore was completed successfully, but the merchant has not received it.', 'Password reset was quick and the account is accessible again.', 'A suspicious card transaction appeared on my statement this morning.']
Sentence count: 11
word tokens ['Arjun', 'paid', 'SGD', '120', 'at', 'Marina', 'Bay', 'yesterday', ',', 'but', 'the', 'card', 'payment', 'was', 'declined', 'twice', '.']


### Regex tokenization

In [8]:
tokenizer = RegexpTokenizer(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+(?:\.\d+)?")
print(tokenizer.tokenize(documents[0]["text"]))

['Arjun', 'paid', 'SGD', '120', 'at', 'Marina', 'Bay', 'yesterday', 'but', 'the', 'card', 'payment', 'was', 'declined', 'twice']


## 4. Multi-word expressions

In [12]:
mwe = MWETokenizer(
    [("credit", "card"), ("bank", "account"), ("mobile", "payment")],
    separator="_"
)
print(mwe.tokenize("someone used my credit card and bank account".split()))

['someone', 'used', 'my', 'credit_card', 'and', 'bank_account']


## 5. Stopword removal

In [11]:
stop_words = set(stopwords.words("english"))
tokens0 = word_tokenize(documents[5]["text"].lower())

filtered = [
    t for t in tokens0
    if t.isalpha() and t not in stop_words
]

print("Original:", tokens0)
print("Filtered:", filtered)

Original: ['someone', 'used', 'my', 'credit', 'card', 'online', '.', 'i', 'did', 'not', 'make', 'this', 'purchase', '.']
Filtered: ['someone', 'used', 'credit', 'card', 'online', 'make', 'purchase']


> Stopword removal is task-dependent. Negators such as **not**, **never**, and **no** may be critical for sentiment or intent.

## 6. Stemming

In [13]:
porter = PorterStemmer()
snowball = SnowballStemmer("english")

words = ["payments", "paying", "transactions", "verification", "verified", "accessible"]

for word in words:
    print(f"{word:15s} Porter={porter.stem(word):12s} Snowball={snowball.stem(word)}")

payments        Porter=payment      Snowball=payment
paying          Porter=pay          Snowball=pay
transactions    Porter=transact     Snowball=transact
verification    Porter=verif        Snowball=verif
verified        Porter=verifi       Snowball=verifi
accessible      Porter=access       Snowball=access


## 7. Lemmatization

In [14]:
lemmatizer = WordNetLemmatizer()

examples = [
    ("transactions", "n"),
    ("running", "v"),
    ("better", "a"),
    ("blocked", "v"),
]

for word, pos in examples:
    print(word, "->", lemmatizer.lemmatize(word, pos=pos))

transactions -> transaction
running -> run
better -> good
blocked -> block


# Part II — Linguistic annotation
## 8. POS tagging

**POS tagging** is an important NLTK capability missing from the original list.

In [15]:
sentence = "The suspicious transaction appeared on my statement this morning."
tagged = pos_tag(word_tokenize(sentence))
pprint(tagged)

[('The', 'DT'),
 ('suspicious', 'JJ'),
 ('transaction', 'NN'),
 ('appeared', 'VBD'),
 ('on', 'IN'),
 ('my', 'PRP$'),
 ('statement', 'NN'),
 ('this', 'DT'),
 ('morning', 'NN'),
 ('.', '.')]


## 9. Chunking / shallow parsing

In [16]:
grammar = r"""
NP: {<DT|PRP\$>?<JJ.*>*<NN.*>+}
"""

chunker = RegexpParser(grammar)
sentence = "The suspicious card transaction appeared on my monthly statement."
tagged = pos_tag(word_tokenize(sentence))

tree = chunker.parse(tagged)
print(tree)

(S
  (NP The/DT suspicious/JJ card/JJ transaction/NN)
  appeared/VBD
  on/IN
  (NP my/PRP$ monthly/JJ statement/NN)
  ./.)


## 10. Named Entity Recognition

In [ ]:
sentence = "Daniel travelled from Singapore to London and contacted Microsoft."
tagged = pos_tag(word_tokenize(sentence))
entities = ne_chunk(tagged)
print(entities)

# Part III — Grammar and parsing
## 11. Context-Free Grammar

In [17]:
grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det Adj N | ProperNoun
VP -> V NP | V
Det -> 'the' | 'a'
Adj -> 'suspicious'
N -> 'customer' | 'transaction' | 'account'
ProperNoun -> 'Daniel'
V -> 'reported' | 'failed'
""")

parser = ChartParser(grammar)
sentence = "Daniel reported a suspicious transaction".split()

for tree in parser.parse(sentence):
    print(tree)

(S
  (NP (ProperNoun Daniel))
  (VP (V reported) (NP (Det a) (Adj suspicious) (N transaction))))


## 12. Constituency parsing
The `ChartParser` above creates a hierarchical constituency tree.

In [18]:
trees = list(parser.parse(sentence))

if trees:
    tree = trees[0]
    print("Height:", tree.height())
    print("Leaves:", tree.leaves())
    print("\nNoun phrases:")
    for subtree in tree.subtrees(lambda t: t.label() == "NP"):
        print(subtree)

Height: 5
Leaves: ['Daniel', 'reported', 'a', 'suspicious', 'transaction']

Noun phrases:
(NP (ProperNoun Daniel))
(NP (Det a) (Adj suspicious) (N transaction))


## 13. PCFG and probabilistic parsing

In [19]:
pcfg = PCFG.fromstring("""
S -> NP VP [1.0]
NP -> ProperNoun [0.30] | Det N [0.40] | Det Adj N [0.30]
VP -> V NP [0.75] | V [0.25]
Det -> 'the' [0.5] | 'a' [0.5]
Adj -> 'suspicious' [1.0]
N -> 'customer' [0.30] | 'transaction' [0.40] | 'account' [0.30]
ProperNoun -> 'Daniel' [1.0]
V -> 'reported' [0.70] | 'failed' [0.30]
""")

viterbi_parser = ViterbiParser(pcfg)

for tree in viterbi_parser.parse(
    "Daniel reported a suspicious transaction".split()
):
    print(tree)
    print("Probability:", tree.prob())

(S
  (NP (ProperNoun Daniel))
  (VP
    (V reported)
    (NP (Det a) (Adj suspicious) (N transaction)))) (p=0.00945)
Probability: 0.009449999999999998


## 14. Dependency-related structures

NLTK provides dependency graphs and interfaces, although it is **not a modern production-grade dependency parser out of the box**. For current dependency parsing, spaCy or Stanza is usually preferable.

In [20]:
dependency_data = """Daniel NNP 2 nsubj
reported VBD 0 root
transaction NN 2 obj
"""

dep_graph = DependencyGraph(dependency_data)

for address, node in dep_graph.nodes.items():
    if address == 0:
        continue
    print({
        "address": address,
        "word": node["word"],
        "tag": node["tag"],
        "head": node["head"],
        "relation": node["rel"],
        "dependents": node["deps"],
    })

{'address': 1, 'word': 'Daniel', 'tag': 'NNP', 'head': 2, 'relation': 'nsubj', 'dependents': defaultdict(<class 'list'>, {})}
{'address': 2, 'word': 'reported', 'tag': 'VBD', 'head': 0, 'relation': 'root', 'dependents': defaultdict(<class 'list'>, {'nsubj': [1], 'obj': [3]})}
{'address': 3, 'word': 'transaction', 'tag': 'NN', 'head': 2, 'relation': 'obj', 'dependents': defaultdict(<class 'list'>, {})}


/home/anirban/polyglot-jupyter/.venv/lib/python3.11/site-packages/nltk/parse/dependencygraph.py:376: UserWarning: The graph doesn't contain a node that depends on the root element.
  warnings.warn(


# Part IV — Lexical semantics
## 15. WordNet

In [21]:
for synset in wn.synsets("bank")[:6]:
    print(synset.name(), "=>", synset.definition())

bank.n.01 => sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 => a financial institution that accepts deposits and channels the money into lending activities
bank.n.03 => a long ridge or pile
bank.n.04 => an arrangement of similar objects in a row or in tiers
bank.n.05 => a supply or stock held in reserve for future use (especially in emergencies)
bank.n.06 => the funds held by a gambling house or the dealer in some gambling games


### WordNet relations

In [22]:
dog = wn.synset("dog.n.01")

print("Synonyms:", dog.lemma_names())
print("Hypernyms:", dog.hypernyms())
print("Some hyponyms:", dog.hyponyms()[:8])

Synonyms: ['dog', 'domestic_dog', 'Canis_familiaris']
Hypernyms: [Synset('canine.n.02'), Synset('domestic_animal.n.01')]
Some hyponyms: [Synset('great_pyrenees.n.01'), Synset('working_dog.n.01'), Synset('hunting_dog.n.01'), Synset('poodle.n.01'), Synset('mexican_hairless.n.01'), Synset('puppy.n.01'), Synset('newfoundland.n.01'), Synset('corgi.n.01')]


## 16. Semantic similarity

In [23]:
dog = wn.synset("dog.n.01")
cat = wn.synset("cat.n.01")
car = wn.synset("car.n.01")

print("dog vs cat:", dog.wup_similarity(cat))
print("dog vs car:", dog.wup_similarity(car))

dog vs cat: 0.8571428571428571
dog vs car: 0.4


## 17. Word Sense Disambiguation — Lesk

In [26]:
examples = [
    "I deposited money into the bank",
    "We sat on the bank of the river",
]

for sentence in examples:
    sense = lesk(word_tokenize(sentence.lower()), "bank")
    print("\n", sentence)
    print("Sense:", sense)
    if sense:
        print("Definition:", sense.definition())


 I deposited money into the bank
Sense: Synset('depository_financial_institution.n.01')
Definition: a financial institution that accepts deposits and channels the money into lending activities

 We sat on the bank of the river
Sense: Synset('bank.n.01')
Definition: sloping land (especially the slope beside a body of water)


# Part V — N-grams and statistical NLP
## 18. N-grams

In [27]:
all_tokens = [
    token.lower()
    for token in word_tokenize(raw_text)
    if token.isalpha()
]

print("Bigrams:", list(bigrams(all_tokens))[:10])
print("Trigrams:", list(trigrams(all_tokens))[:10])
print("4-grams:", list(ngrams(all_tokens, 4))[:5])

Bigrams: [('arjun', 'paid'), ('paid', 'sgd'), ('sgd', 'at'), ('at', 'marina'), ('marina', 'bay'), ('bay', 'yesterday'), ('yesterday', 'but'), ('but', 'the'), ('the', 'card'), ('card', 'payment')]
Trigrams: [('arjun', 'paid', 'sgd'), ('paid', 'sgd', 'at'), ('sgd', 'at', 'marina'), ('at', 'marina', 'bay'), ('marina', 'bay', 'yesterday'), ('bay', 'yesterday', 'but'), ('yesterday', 'but', 'the'), ('but', 'the', 'card'), ('the', 'card', 'payment'), ('card', 'payment', 'was')]
4-grams: [('arjun', 'paid', 'sgd', 'at'), ('paid', 'sgd', 'at', 'marina'), ('sgd', 'at', 'marina', 'bay'), ('at', 'marina', 'bay', 'yesterday'), ('marina', 'bay', 'yesterday', 'but')]


## 19. Frequency analysis

In [28]:
fdist = FreqDist(all_tokens)
print(fdist.most_common(15))
print("card frequency:", fdist["card"])

[('the', 8), ('card', 4), ('was', 3), ('not', 3), ('account', 3), ('and', 3), ('my', 3), ('but', 2), ('payment', 2), ('after', 2), ('password', 2), ('transaction', 2), ('i', 2), ('is', 2), ('this', 2)]
card frequency: 4


## 20. Conditional frequency distributions

In [29]:
cfd = ConditionalFreqDist(
    (doc["label"], token.lower())
    for doc in documents
    for token in word_tokenize(doc["text"])
    if token.isalpha()
)

for label in ["payment", "account", "fraud"]:
    print(label, "=>", cfd[label].most_common(8))

payment => [('the', 5), ('but', 2), ('payment', 2), ('was', 2), ('arjun', 1), ('paid', 1), ('sgd', 1), ('at', 1)]
account => [('account', 3), ('the', 2), ('password', 2), ('is', 2), ('and', 2), ('priya', 1), ('can', 1), ('not', 1)]
fraud => [('card', 3), ('transaction', 2), ('my', 2), ('this', 2), ('daniel', 1), ('noticed', 1), ('an', 1), ('unauthorized', 1)]


## 21. Collocation detection with PMI

In [30]:
finder = BigramCollocationFinder.from_words(all_tokens)
finder.apply_freq_filter(2)

scores = finder.score_ngrams(BigramAssocMeasures.pmi)

for pair, score in scores[:15]:
    print(pair, round(score, 3))

('account', 'is') 5.129
('but', 'the') 3.714
('the', 'card') 2.714


### Trigram collocations

In [31]:
tri_finder = TrigramCollocationFinder.from_words(all_tokens)
scores = tri_finder.score_ngrams(TrigramAssocMeasures.likelihood_ratio)

for phrase, score in scores[:10]:
    print(phrase, round(score, 3))

('account', 'is', 'accessible') 24.512
('account', 'is', 'locked') 24.512
('bank', 'account', 'is') 24.512
('access', 'her', 'savings') 22.597
('accessible', 'again', 'a') 22.597
('again', 'a', 'suspicious') 22.597
('app', 'excellent', 'support') 22.597
('arjun', 'paid', 'sgd') 22.597
('at', 'marina', 'bay') 22.597
('daniel', 'noticed', 'an') 22.597


## 22. Statistical N-gram language modelling

For an order-\(n\) language model:

\[
P(w_t \mid w_{t-n+1}, \ldots, w_{t-1})
\]

We compare MLE, Laplace smoothing, and Kneser-Ney interpolation.

In [32]:
tokenized_sentences = [
    [t.lower() for t in word_tokenize(doc["text"]) if t.isalpha()]
    for doc in documents
]

order = 2
train_data, vocab = padded_everygram_pipeline(order, tokenized_sentences)

mle = MLE(order)
mle.fit(train_data, vocab)

print("P(payment | mobile) =", mle.score("payment", ["mobile"]))

P(payment | mobile) = 1.0


### Laplace smoothing

In [33]:
train_data, vocab = padded_everygram_pipeline(2, tokenized_sentences)
laplace = Laplace(2)
laplace.fit(train_data, vocab)

print("Smoothed P(payment | mobile) =", laplace.score("payment", ["mobile"]))

Smoothed P(payment | mobile) = 0.024691358024691357


### Kneser-Ney interpolation

In [34]:
train_data, vocab = padded_everygram_pipeline(3, tokenized_sentences)
kn = KneserNeyInterpolated(3)
kn.fit(train_data, vocab)

print(
    "P(transaction | suspicious card) =",
    kn.score("transaction", ["suspicious", "card"])
)

P(transaction | suspicious card) = 0.9226785714285715


# Part VI — Corpus exploration
## 23. Concordance

In [35]:
support_text = Text(all_tokens)
support_text.concordance("card", width=80, lines=10)

Displaying 4 of 4 matches:
d sgd at marina bay yesterday but the card payment was declined twice priya can 
transaction from london and wants the card blocked immediately the mobile paymen
 keeps failing someone used my credit card online i did not make this purchase t
ount is accessible again a suspicious card transaction appeared on my statement 


## 24. Lexical diversity

In [36]:
token_count = len(all_tokens)
vocabulary_size = len(set(all_tokens))

print("Tokens:", token_count)
print("Vocabulary:", vocabulary_size)
print("Lexical diversity:", vocabulary_size / token_count)

Tokens: 105
Vocabulary: 77
Lexical diversity: 0.7333333333333333


## 25. Built-in corpora

In [37]:
from nltk.corpus import brown, reuters, gutenberg

print("Brown categories:", brown.categories())
print("\nReuters categories:", reuters.categories()[:20])
print("\nGutenberg files:", gutenberg.fileids())

Brown categories: ['adventure', 'belles_lettres', 'editorial', 'fiction', 'government', 'hobbies', 'humor', 'learned', 'lore', 'mystery', 'news', 'religion', 'reviews', 'romance', 'science_fiction']

Reuters categories: ['acq', 'alum', 'barley', 'bop', 'carcass', 'castor-oil', 'cocoa', 'coconut', 'coconut-oil', 'coffee', 'copper', 'copra-cake', 'corn', 'cotton', 'cotton-oil', 'cpi', 'cpu', 'crude', 'dfl', 'dlr']

Gutenberg files: ['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


## 26. Corpus processing

In [38]:
news_words = [
    word.lower()
    for word in brown.words(categories="news")
    if word.isalpha()
]

print(FreqDist(news_words).most_common(20))

[('the', 6386), ('of', 2861), ('and', 2186), ('to', 2144), ('a', 2130), ('in', 2020), ('for', 969), ('that', 829), ('is', 733), ('was', 717), ('on', 691), ('he', 642), ('at', 636), ('with', 567), ('be', 526), ('as', 517), ('by', 504), ('it', 478), ('his', 428), ('said', 406)]


# Part VII — Additional capabilities you missed
## 27. VADER sentiment analysis

In [39]:
sia = SentimentIntensityAnalyzer()

for text in [
    "Excellent support! The issue was resolved immediately.",
    "The app is fine.",
    "Terrible service. My payment keeps failing.",
]:
    print(text)
    print(sia.polarity_scores(text))
    print()

Excellent support! The issue was resolved immediately.
{'neg': 0.0, 'neu': 0.323, 'pos': 0.677, 'compound': 0.8122}

The app is fine.
{'neg': 0.0, 'neu': 0.625, 'pos': 0.375, 'compound': 0.2023}

Terrible service. My payment keeps failing.
{'neg': 0.615, 'neu': 0.385, 'pos': 0.0, 'compound': -0.7506}



## 28. Edit distance

In [40]:
from nltk.metrics.distance import edit_distance

for a, b in [
    ("transaction", "transation"),
    ("payment", "paymant"),
    ("account", "acount"),
]:
    print(a, b, "=>", edit_distance(a, b))

transaction transation => 1
payment paymant => 1
account acount => 1


## 29. Classical text classification

In [41]:
from nltk.classify import NaiveBayesClassifier

def document_features(text):
    words = {
        token.lower()
        for token in word_tokenize(text)
        if token.isalpha()
    }

    keywords = [
        "card", "payment", "account", "password",
        "transaction", "fraud", "blocked", "login",
    ]

    return {
        f"contains({keyword})": keyword in words
        for keyword in keywords
    }

training_set = [
    (document_features(doc["text"]), doc["label"])
    for doc in documents
]

classifier = NaiveBayesClassifier.train(training_set)

test_text = "My card shows an unauthorized transaction"

print("Prediction:", classifier.classify(document_features(test_text)))
classifier.show_most_informative_features(10)

Prediction: fraud
Most Informative Features
          contains(card) = True            fraud : paymen =      2.3 : 1.0
      contains(password) = False           fraud : accoun =      2.3 : 1.0
       contains(payment) = False          accoun : paymen =      2.3 : 1.0
   contains(transaction) = False          accoun : fraud  =      2.3 : 1.0
       contains(blocked) = False          accoun : fraud  =      1.4 : 1.0
          contains(card) = False          accoun : paymen =      1.4 : 1.0
         contains(login) = False           fraud : accoun =      1.4 : 1.0
       contains(account) = False           fraud : paymen =      1.0 : 1.0


## 30. Classical sequence tagging

In [49]:
from nltk.corpus import treebank
from nltk.tag import UnigramTagger

tagged_sentences = treebank.tagged_sents()
print(tagged_sentences)
split = int(len(tagged_sentences) * 0.8)

tagger = UnigramTagger(tagged_sentences[:split])
print("\n",tagger)

print("\n",
    tagger.tag(
        "the customer reported a transaction".split()
    )
)

print("\n","Accuracy:", tagger.accuracy(tagged_sentences[split:]))

[[('Pierre', 'NNP'), ('Vinken', 'NNP'), (',', ','), ('61', 'CD'), ('years', 'NNS'), ('old', 'JJ'), (',', ','), ('will', 'MD'), ('join', 'VB'), ('the', 'DT'), ('board', 'NN'), ('as', 'IN'), ('a', 'DT'), ('nonexecutive', 'JJ'), ('director', 'NN'), ('Nov.', 'NNP'), ('29', 'CD'), ('.', '.')], [('Mr.', 'NNP'), ('Vinken', 'NNP'), ('is', 'VBZ'), ('chairman', 'NN'), ('of', 'IN'), ('Elsevier', 'NNP'), ('N.V.', 'NNP'), (',', ','), ('the', 'DT'), ('Dutch', 'NNP'), ('publishing', 'VBG'), ('group', 'NN'), ('.', '.')], ...]

 <UnigramTagger: size=11044>

 [('the', 'DT'), ('customer', 'NN'), ('reported', 'VBD'), ('a', 'DT'), ('transaction', 'NN')]

 Accuracy: 0.8608213982733669


## 31. Evaluation metrics

In [46]:
from nltk.metrics import precision, recall, f_measure, ConfusionMatrix

gold = ["fraud", "account", "payment", "fraud", "payment"]
pred = ["fraud", "payment", "payment", "fraud", "account"]

print(ConfusionMatrix(gold, pred))

for label in sorted(set(gold) | set(pred)):
    gold_set = {i for i, y in enumerate(gold) if y == label}
    pred_set = {i for i, y in enumerate(pred) if y == label}

    print(
        label,
        "precision =", precision(gold_set, pred_set),
        "recall =", recall(gold_set, pred_set),
        "f1 =", f_measure(gold_set, pred_set),
    )

        | a   p |
        | c   a |
        | c f y |
        | o r m |
        | u a e |
        | n u n |
        | t d t |
--------+-------+
account |<.>. 1 |
  fraud | .<2>. |
payment | 1 .<1>|
--------+-------+
(row = reference; col = test)

account precision = 0.0 recall = 0.0 f1 = 0
fraud precision = 1.0 recall = 1.0 f1 = 1.0
payment precision = 0.5 recall = 0.5 f1 = 0.5


## 32. BLEU

In [44]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [["the", "card", "payment", "was", "declined"]]
candidate = ["the", "card", "transaction", "was", "declined"]

score = sentence_bleu(
    reference,
    candidate,
    smoothing_function=SmoothingFunction().method1,
)

print("BLEU:", score)

BLEU: 0.16068568378893033


## 33. METEOR

In [45]:
from nltk.translate.meteor_score import meteor_score

reference = [["the", "payment", "was", "completed", "successfully"]]
candidate = ["the", "transaction", "completed", "successfully"]

print("METEOR:", meteor_score(reference, candidate))

METEOR: 0.5215419501133786


# Part VIII — What else NLTK can do

Your original list was already broad. Important capabilities worth explicitly adding are:

| Capability | NLTK support |
|---|---|
| **POS tagging** | pretrained POS tagger; unigram/bigram/trigram/HMM/Brill taggers |
| **Sentence segmentation** | Punkt models |
| **Regex tokenization** | `RegexpTokenizer` |
| **Multi-word-expression tokenization** | `MWETokenizer` |
| **Concordance analysis** | `Text.concordance()` |
| **Similar/common contexts** | `Text.similar()`, `common_contexts()` |
| **Lexical dispersion** | `Text.dispersion_plot()` |
| **Conditional frequency distributions** | `ConditionalFreqDist` |
| **Sentiment analysis** | VADER |
| **Edit/string distance** | edit distance and related distance metrics |
| **Custom/classical tagger training** | unigram, bigram, trigram, regex, HMM, Brill |
| **Evaluation** | precision, recall, F1, confusion matrices |
| **MT/generation evaluation** | BLEU, METEOR, NIST, GLEU, chrF |
| **Tree manipulation** | `Tree`, subtree traversal |
| **Corpus readers** | plain, categorized, tagged and parsed corpus interfaces |
| **Probability distributions** | frequency/probability distribution abstractions |
| **Language-model smoothing** | Laplace, Lidstone, Witten-Bell, Kneser-Ney |
| **Semantic similarity** | path, Wu-Palmer, Resnik, Lin, Jiang-Conrath |
| **Association statistics** | PMI, chi-square, likelihood ratio, Dice, Jaccard |
| **Alignment/translation utilities** | classical alignment and IBM-model-related tools |

# Part IX — Recommended learning sequence

```text
RAW TEXT
   │
   ├── Sentence segmentation
   ├── Tokenization
   ├── Stopword filtering
   ├── Stemming / Lemmatization
   │
   ▼
LEXICAL ANALYSIS
   ├── Frequency distributions
   ├── N-grams
   ├── Collocations
   ├── WordNet
   └── Word Sense Disambiguation
   │
   ▼
LINGUISTIC ANNOTATION
   ├── POS tagging
   ├── Chunking
   └── Named Entity Recognition
   │
   ▼
SYNTACTIC STRUCTURE
   ├── CFG
   ├── Constituency parsing
   ├── PCFG
   ├── Probabilistic parsing
   └── Dependency structures
   │
   ▼
STATISTICAL NLP
   ├── Conditional frequencies
   ├── N-gram language models
   ├── Smoothing
   └── Classical classification
   │
   ▼
EVALUATION / CORPUS ANALYSIS
   ├── Precision / Recall / F1
   ├── BLEU / METEOR
   ├── Concordance
   └── Corpus readers
```

# Part X — Where NLTK fits today

| Task | NLTK | spaCy | Hugging Face |
|---|---:|---:|---:|
| NLP education | ★★★★★ | ★★★ | ★★★ |
| Corpus linguistics | ★★★★★ | ★★ | ★ |
| WordNet / lexical semantics | ★★★★★ | ★★ | ★ |
| CFG / PCFG parsing | ★★★★★ | ★ | ★ |
| Classical statistical NLP | ★★★★★ | ★★ | ★★ |
| Production POS / NER | ★★ | ★★★★★ | ★★★★★ |
| Dependency parsing | ★★ | ★★★★★ | ★★★★ |
| Transformers | ★ | ★★★ | ★★★★★ |
| Embeddings | ★ | ★★★ | ★★★★★ |

### Takeaway

Think of NLTK as a **computational-linguistics laboratory**. It exposes the mechanics of words, tags, chunks, grammars, parse trees, probability distributions, lexical graphs, corpora, and classical statistical NLP in a way that modern end-to-end neural libraries often abstract away.